In [1]:
# ============================================================
# Mini-projet Data Science – Prédiction du churn (Option A)
# ============================================================
# Objectif : construire un mini-projet complet à partir d’un seul fichier :
#   - Chargement des données
#   - Analyse exploratoire (EDA)
#   - Nettoyage & préparation
#   - Séparation train / test
#   - Construction d’un modèle de churn
#   - Évaluation (métriques classiques)
#   - Intégration d’un outil avancé de qualité : Great Expectations
#
# Remarque importante :
#   - Adapter le chemin vers "data_churn.csv" si besoin.
#   - On commence pqr instqller Great Expectations aue nous utiliserons :
# ============================================================

In [ ]:
pip uninstall great_expectations

In [21]:
pip install "great_expectations==0.18.21"

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import great_expectations as ge
print(ge.__version__)

In [4]:
# =========================
# 0. Imports principaux
# =========================
import great_expectations as gx
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Configurqtion pour rendre les tableaux plus lisibles
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [5]:
# =================================
# 1. Chargement des données brutes
# =================================

# Commentaire :
#   - On part d’un seul fichier "data_churn.csv" contenant toutes les observations.
#   - On le charge dans un DataFrame Pandas.
#   - Ensuite, on vérifie rapidement sa structure (taille, types, aperçu).

data_path = "data_churn.csv"  # adapter le chemin si nécessaire
df = pd.read_csv(data_path)

print("Aperçu des 5 premières lignes :")
print(df.head())

print("\nDimensions du dataset (n_lignes, n_colonnes) :")
print(df.shape)

print("\nTypes des colonnes et présence de valeurs non nulles (df.info()) :")
print(df.info())

Aperçu des 5 premières lignes :
   CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  Payment Delay Subscription Type Contract Length  \
0         2.0  30.0  Female    39.0             14.0            5.0           18.0          Standard          Annual   
1         3.0  65.0  Female    49.0              1.0           10.0            8.0             Basic         Monthly   
2         4.0  55.0  Female    14.0              4.0            6.0           18.0             Basic       Quarterly   
3         5.0  58.0    Male    38.0             21.0            7.0            7.0          Standard         Monthly   
4         6.0  23.0    Male    32.0             20.0            5.0            8.0             Basic         Monthly   

   Total Spend  Last Interaction  Churn  
0        932.0              17.0    1.0  
1        557.0               6.0    1.0  
2        185.0               3.0    1.0  
3        396.0              29.0    1.0  
4        617.0              20.0    1

In [6]:
# ===========================================
# 2. Analyse exploratoire simple (EDA)
# ===========================================
# Ici, on montre explicitement :
#   - Comment lister les colonnes
#   - Comment repérer les valeurs manquantes
#   - Comment résumer les variables numériques
#   - Comment explorer les variables catégorielles et la variable cible


# 2.1. Lister les colonnes
print("\nListe des colonnes du dataset :")
print(df.columns.tolist())


# 2.2. Vérifier les valeurs manquantes
print("\nNombre de valeurs manquantes par colonne (df.isna().sum()) :")
print(df.isna().sum())

print("\nProportion de valeurs manquantes par colonne (df.isna().mean()) :")
print(df.isna().mean())


# 2.3. Résumé statistique des variables numériques
# Pandas détecte automatiquement les colonnes numériques
print("\nRésumé statistique des variables numériques (df.describe()) :")
print(df.describe())


# 2.4. Identification des variables catégorielles
# On va repérer les colonnes de type 'object'
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()
print("\nColonnes catégorielles détectées :")
print(categorical_cols)

# Pour chaque variable catégorielle, on affiche la distribution
for col in categorical_cols:
    print(f"\nDistribution de la variable catégorielle '{col}' (value_counts):")
    print(df[col].value_counts(dropna=False))
    print("\nDistribution en pourcentage :")
    print(df[col].value_counts(normalize=True, dropna=False))


# 2.5. Distribution de la variable cible "Churn"
print("\nDistribution brute de la cible 'Churn' :")
print(df["Churn"].value_counts(dropna=False))

print("\nDistribution en pourcentage de 'Churn' :")
print(df["Churn"].value_counts(normalize=True, dropna=False))


Liste des colonnes du dataset :
['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']

Nombre de valeurs manquantes par colonne (df.isna().sum()) :
CustomerID           1
Age                  1
Gender               1
Tenure               1
Usage Frequency      1
Support Calls        1
Payment Delay        1
Subscription Type    1
Contract Length      1
Total Spend          1
Last Interaction     1
Churn                1
dtype: int64

Proportion de valeurs manquantes par colonne (df.isna().mean()) :
CustomerID           0.000002
Age                  0.000002
Gender               0.000002
Tenure               0.000002
Usage Frequency      0.000002
Support Calls        0.000002
Payment Delay        0.000002
Subscription Type    0.000002
Contract Length      0.000002
Total Spend          0.000002
Last Interaction     0.000002
Churn                0.000002
dtype: flo

In [7]:
# ===================================================
# 3. Nettoyage simple des données (data cleaning)
# ===================================================
# Ici, le dataset contient une ligne avec des valeurs manquantes (NA) dans chaque colonne.
# Comme les NA sont très rares, la stratégie simple consiste à supprimer les lignes incomplètes.
# En pratique, on pourrait aussi :
#   - Imputer les valeurs manquantes
#   - Analyser précisément les causes des NA
#   - Mettre en place des tests de qualité (qu’on fera plus loin via Great Expectations)

df_clean = df.dropna().copy()

print("\nDimensions après suppression des lignes avec NA :")
print(df_clean.shape)

print("\nVérification des valeurs manquantes après nettoyage :")
print(df_clean.isna().sum())


Dimensions après suppression des lignes avec NA :
(440832, 12)

Vérification des valeurs manquantes après nettoyage :
CustomerID           0
Age                  0
Gender               0
Tenure               0
Usage Frequency      0
Support Calls        0
Payment Delay        0
Subscription Type    0
Contract Length      0
Total Spend          0
Last Interaction     0
Churn                0
dtype: int64


In [8]:
# =========================================================
# 4. Séparation entraînement / test (train_test_split)
# =========================================================
# Nous allons :
#   - Définir la variable cible : "Churn"
#   - Définir X (features) = toutes les autres colonnes
#   - Faire un split 80% / 20% avec stratification sur la cible
#     pour conserver le même taux de churn dans train et test.

target = "Churn"
X = df_clean.drop(columns=[target])
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y  # très important pour garder le même ratio 0/1
)

print("\nTaille du jeu de train : ", X_train.shape)
print("Taille du jeu de test  : ", X_test.shape)

print("\nTaux de churn dans le jeu de train : ", y_train.mean())
print("Taux de churn dans le jeu de test  : ", y_test.mean())


 (352665, 11) de train : 
Taille du jeu de test  :  (88167, 11)

Taux de churn dans le jeu de train :  0.5671075950264415
Taux de churn dans le jeu de test  :  0.5671056064060249


In [9]:
# =========================================================
# 5. Préparation des features (numériques + catégorielles)
# =========================================================
# On sépare les colonnes en deux groupes :
#   - Features numériques : standardisation (StandardScaler)
#   - Features catégorielles : encodage one-hot (OneHotEncoder)
#
# Les colonnes sont connues à partir du dataset :
#   - Numériques : Age, Tenure, Usage Frequency, Support Calls,
#                  Payment Delay, Total Spend, Last Interaction
#   - Catégorielles : Gender, Subscription Type, Contract Length

numeric_features = [
    "Age",
    "Tenure",
    "Usage Frequency",
    "Support Calls",
    "Payment Delay",
    "Total Spend",
    "Last Interaction",
]

categorical_features = [
    "Gender",
    "Subscription Type",
    "Contract Length",
]

# Définition du transformeur de colonnes
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop"  # on ignore les autres colonnes (CustomerID)
)

In [10]:
# ===============================================
# 6. Modélisation : régression logistique
# ===============================================
# On crée un Pipeline sklearn qui :
#   - applique d’abord le preprocessing (StandardScaler + OneHotEncoder)
#   - puis entraîne un modèle de régression logistique
#
# Les qvantages de ce Pipeline sont :
#   - Code plus propre et modulaire
#   - Reproductibilité du preprocessing
#   - Moins de risques d’erreur entre train et test

log_reg_clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=1000)),
    ]
)

# Entraînement
log_reg_clf.fit(X_train, y_train)

# Prédictions sur le jeu de test
y_pred = log_reg_clf.predict(X_test)
y_proba = log_reg_clf.predict_proba(X_test)[:, 1]  # probas pour la classe 1 (churn)

C:\Users\ANGNECHEKO\anaconda3\envs\DataScience\Lib\site-packages\sklearn\linear_model\_logistic.py:451: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  opt_res = optimize.minimize(


In [11]:
# ===============================================
# 7. Évaluation du modèle
# ===============================================
# On calcule plusieurs métriques :
#   - Classification report (precision, recall, f1-score)
#   - Matrice de confusion
#   - AUC ROC (mesure globale de la qualité de classement des probabilités)

print("\n=== Rapport de classification (régression logistique) ===")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("Matrice de confusion :")
print(cm)

roc_auc = roc_auc_score(y_test, y_proba)
print("\nAUC ROC du modèle : ", roc_auc)


=== Rapport de classification (régression logistique) ===
              precision    recall  f1-score   support

         0.0       0.86      0.90      0.88     38167
         1.0       0.92      0.89      0.90     50000

    accuracy                           0.89     88167
   macro avg       0.89      0.89      0.89     88167
weighted avg       0.89      0.89      0.89     88167

Matrice de confusion :
[[34492  3675]
 [ 5726 44274]]

AUC ROC du modèle :  0.959026233657348


In [12]:
# ===============================================
# 8. Analyse rapide des erreurs
# ===============================================
# On peut identifier :
#   - Faux positifs (prédit churn mais en réalité non)
#   - Faux négatifs (prédit non churn mais en réalité churn)
#
# Cela peut aider à comprendre où le modèle se trompe.

test_results = X_test.copy()
test_results["y_true"] = y_test
test_results["y_pred"] = y_pred
test_results["y_proba"] = y_proba

false_positives = test_results[(test_results["y_true"] == 0) & (test_results["y_pred"] == 1)]
false_negatives = test_results[(test_results["y_true"] == 1) & (test_results["y_pred"] == 0)]

print("\nNombre de faux positifs : ", len(false_positives))
print("Nombre de faux négatifs : ", len(false_negatives))


Nombre de faux positifs :  3675
Nombre de faux négatifs :  5726


In [ ]:
# =========================================================
# 9. Intégration d’un outil avancé : Great Expectations
# =========================================================
# Notre objectif ici est de :
#   - Définir quelques règles de qualité sur le dataset de churn
#   - Vérifier que les données respectent ces règles
#   - Ceci fait partie du "référentiel de qualité" data
#
# Attention :
#   # On convertit le DataFrame Pandas en DataFrame Great Expectations
import great_expectations as gx
ge_df = gx.from_pandas(df_clean)

# Exemples d’attentes (expectations) sur les données :
# 1) Colonnes numériques non nulles
for col in numeric_features:
    ge_df.expect_column_values_to_not_be_null(col)

# 2) Variables catégorielles non nulles
for col in categorical_features:
    ge_df.expect_column_values_to_not_be_null(col)

# 3) Domaines de valeurs raisonnables pour quelques variables
ge_df.expect_column_values_to_be_between("Age", min_value=18, max_value=100)
ge_df.expect_column_values_to_be_between("Tenure", min_value=0, max_value=100)
ge_df.expect_column_values_to_be_between("Usage Frequency", min_value=0, max_value=100)
ge_df.expect_column_values_to_be_between("Support Calls", min_value=0, max_value=50)
ge_df.expect_column_values_to_be_between("Payment Delay", min_value=0, max_value=60)
ge_df.expect_column_values_to_be_between("Total Spend", min_value=0, max_value=10000)
ge_df.expect_column_values_to_be_between("Last Interaction", min_value=0, max_value=365)

# 4) Valeurs autorisées pour les colonnes catégorielles
ge_df.expect_column_values_to_be_in_set(
    "Gender",
    value_set=["Male", "Female"]
)
ge_df.expect_column_values_to_be_in_set(
    "Subscription Type",
    value_set=["Basic", "Standard", "Premium"]
)
ge_df.expect_column_values_to_be_in_set(
    "Contract Length",
    value_set=["Monthly", "Quarterly", "Annual"]
)

# Validation globale
validation_result = ge_df.validate()

print("\n=== Résultat de la validation Great Expectations ===")
print("Success global :", validation_result["success"])

# (Optionnel) afficher un résumé des expectations
for res in validation_result["results"][:10]:
    print(res["expectation_config"]["expectation_type"],
        "- success =", res["success"] )

In [ ]:
# =========================================================
# 10. Conclusion (à rédiger en Markdown dans le notebook)
# =========================================================
# Dans le notebook final, on pourra ajouter une cellule Markdown avec :
#   - Résumé de l’EDA :
#       * Profil des clients (âge, tenure, usage, etc.)
#       * Taux de churn global (~56-57 %)
#       * Différences de churn selon abonnement / contrat (à analyser)
#   - Résumé du modèle :
#       * Modèle utilisé : régression logistique
#       * Pipeline avec standardisation + encodage one-hot
#       * Performance : rapport de classification + AUC ROC
#   - Référentiel de qualité :
#       * Critères data (NA, plage de valeurs, cohérence)
#       * Critères modèle (métriques, robustesse, fairness à discuter)
#       * Critères de reproductibilité (pipeline, random_state, etc.)
#   - Auto-audit :
#       * Points forts du livrable
#       * Limites (features simples, modèle de base)
#       * Risques en production (drift, biais de sélection, etc.)
#       * Pistes d’amélioration (modèles plus puissants, features engineering,
#         monitoring en production, etc.)
#
# Pour le moment, tout ce qui précède est rédigé en commentaires afin
# d’être facilement copié-collé dans un notebook Jupyter.